In [1]:
import pandas as pd
import tqdm, os
import pickle
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_frmri_articulatory_features

In [2]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_frmri.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[1])

In [3]:
df

,phone1_start,phone1_end,phone2_start,phone2_end,phone1,phone2,context,word,word_id,art_f,ac_f,subject
0,2.647,2.717,2.717,2.907,f,i,f,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
1,2.717,2.907,2.907,2.937,i,l,f_i,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
2,2.907,2.937,2.937,3.067,l,u,f_i_l,filou,2,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
3,3.097,3.157,3.157,3.237,l,a,l,la,4,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
4,3.237,3.317,3.317,3.377,f,ʁ,f,fripouille,5,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/VIDEO_P6...,/Volumes/Transcend/FR-MRI/P6/OTHER/S1/DENOISED...,P6
...,...,...,...,...,...,...,...,...,...,...,...,...
22432,33.071,33.131,33.131,33.211,ʁ,e,p_ʁ,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22433,33.131,33.211,33.211,33.391,e,s,p_ʁ_e,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22434,33.211,33.391,33.391,33.671,s,i,p_ʁ_e_s,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4
22435,33.391,33.671,33.671,33.801,i,z,p_ʁ_e_s_i,précises,78,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/VIDEO_P...,/Volumes/Transcend/FR-MRI/P4/OTHER/S12/DENOISE...,P4


In [ ]:
tokens = {
    "start": [],
    "end": [],
    "syllable": [],
    "art_f": [],
    "ac_f": [],
    "subject": [],
}

metadata = set(
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable1_start"], df["syllable1_end"], df["syllable1"], df["art_f"], df["ac_f"], df["subject"])] + 
    [(start, end, target, art_f, ac_f, subject) for start, end, target, art_f, ac_f, subject in zip(df["syllable2_start"], df["syllable2_end"], df["syllable2"], df["art_f"], df["ac_f"], df["subject"])]
)

metadata = sorted(list(metadata), key=lambda x: (x[3], x[4], x[5]))

for start, end, target, art_f, ac_f, subject in metadata:
    tokens["start"].append(start)
    tokens["end"].append(end)
    tokens["syllable"].append(target)
    tokens["art_f"].append(art_f)
    tokens["ac_f"].append(ac_f)
    tokens["subject"].append(subject)

In [5]:
token_df = pd.DataFrame(tokens)

In [6]:
token_df

,start,end,phone,art_f,ac_f,subject
0,17.177,17.247,d,/Volumes/Transcend/FR-MRI/P7/OTHER/S6/VIDEO_P7...,/Volumes/Transcend/FR-MRI/P7/OTHER/S6/DENOISED...,P7
1,20.618,20.668,ɛ,/Volumes/Transcend/FR-MRI/P9/OTHER/S11/VIDEO_P...,/Volumes/Transcend/FR-MRI/P9/OTHER/S11/DENOISE...,P9
2,31.072,31.132,ɛ,/Volumes/Transcend/FR-MRI/P10/OTHER/S8/VIDEO_P...,/Volumes/Transcend/FR-MRI/P10/OTHER/S8/DENOISE...,P10
3,23.886,24.066,ʁ,/Volumes/Transcend/FR-MRI/P3/OTHER/S14/VIDEO_P...,/Volumes/Transcend/FR-MRI/P3/OTHER/S14/DENOISE...,P3
4,29.121,29.211,o,/Volumes/Transcend/FR-MRI/P2/OTHER/S6/VIDEO_P2...,/Volumes/Transcend/FR-MRI/P2/OTHER/S6/DENOISED...,P2
...,...,...,...,...,...,...
30593,34.890,35.020,d,/Volumes/Transcend/FR-MRI/P8/OTHER/S2/VIDEO_P8...,/Volumes/Transcend/FR-MRI/P8/OTHER/S2/DENOISED...,P8
30594,24.349,24.429,a,/Volumes/Transcend/FR-MRI/P8/OTHER/S4/VIDEO_P8...,/Volumes/Transcend/FR-MRI/P8/OTHER/S4/DENOISED...,P8
30595,17.719,17.789,a,/Volumes/Transcend/FR-MRI/P8/OTHER/S6/VIDEO_P8...,/Volumes/Transcend/FR-MRI/P8/OTHER/S6/DENOISED...,P8
30596,31.439,31.539,e,/Volumes/Transcend/FR-MRI/P8/OTHER/S8/VIDEO_P8...,/Volumes/Transcend/FR-MRI/P8/OTHER/S8/DENOISED...,P8


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import VideoAutoencoder, SequenceAutoencoder

In [8]:
def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

def visualize(clean, recon, name="Reconstruction"):
    import matplotlib.pyplot as plt

    try:
        # Get both channels, frame 0 of first batch element
        clean_mri= clean[0, 0, 0].cpu().numpy()
    except RuntimeError:
        clean_mri = clean[0, 0, 0].detach().cpu().numpy()

    recon_mri = recon[0, 0, 0].detach().cpu().numpy()

    plt.figure(figsize=(10, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_mri, cmap='gray')
    plt.title("Original (MRI)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(recon_mri, cmap='gray')
    plt.title(f"{name} (MRI)")
    plt.axis('off')

    plt.tight_layout()
    plt.savefig(f"video_visualization/frmri/{name.lower()}.png", dpi=300)
    plt.close("all")

def combined_loss(y_true, y_pred, warped_x, criterion, plot=False, ratio=1):
    diff_true = y_true[:, :, 1:, :, :] - y_true[:, :, :-1, :, :]
    diff_pred = y_pred[:, :, 1:, :, :] - y_pred[:, :, :-1, :, :]
    loss = criterion(diff_true, diff_pred) * (1 - ratio) + criterion(y_true, y_pred) * ratio

    if plot:
        visualize(diff_true, diff_pred, name="Reconstruction (differential)")
        visualize(y_true, y_pred, name="Reconstruction")
        visualize(y_true, warped_x, name="Transformation")

    return loss

In [9]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [14]:
log = {
    "batch_idx": [],
    "loss": [],
}

art_model = VideoAutoencoder(
    input_shape=(5, 136, 136),
    latent_dim=1024,
    use_mlp=True,
    grid_sample_cpu_fallback=True,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_frmri_articulatory_features(art_f, start, end)
    if articulatory_features.shape[-2:]!=(136, 136):
        continue
    if articulatory_features is None:
        continue

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    try:
        art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    except RuntimeError:
        continue
    batch.clear()

    # ===== Articulatory =====
    art_out, art_warped_x, art_z = art_model(art_batch, return_warped_x=True, return_latent=True)
    art_loss = combined_loss(art_batch, art_out, art_warped_x, art_criterion, plot=batch_idx % 30 == 0, ratio=0.5)

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/frmri/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/frmri/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/frmri/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/frmri/art_model_training_log.csv", index=False)

300: ART 0.3355:   4%|▍         | 1200/30598 [02:28<57:20,  8.54it/s]  

[ART] latent var mean=8.2897e+04, min=7.3986e-03
[ART] input std=3.7416e-01, recon std=8.8942e-02


600: ART 0.3500:   8%|▊         | 2409/30598 [05:36<1:38:24,  4.77it/s]

[ART] latent var mean=1.0582e+05, min=9.9631e-01
[ART] input std=4.7592e-01, recon std=4.1980e-02


900: ART 0.3386:  12%|█▏        | 3611/30598 [08:49<56:12,  8.00it/s]  

...

7200: ART 0.3347:  94%|█████████▍| 28899/30598 [1:02:48<03:30,  8.06it/s]

[ART] latent var mean=3.8783e+08, min=1.3280e+02
[ART] input std=5.2553e-01, recon std=1.5963e-01


7500: ART 0.3295:  98%|█████████▊| 30102/30598 [1:05:25<01:02,  7.93it/s]

[ART] latent var mean=2.7761e+08, min=7.4498e+00
[ART] input std=3.6293e-01, recon std=9.7558e-02


7622: ART 0.3287: 100%|██████████| 30598/30598 [1:06:29<00:00,  7.67it/s]


In [12]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

In [13]:
BATCH_SIZE = 4

In [ ]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/frmri/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/frmri/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/frmri/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/frmri/ac_model_training_log.csv", index=False)

999: AC 396.5454:   3%|▎         | 996/30598 [00:08<02:31, 195.93it/s] 

[AC ] latent var mean=1.0226e+04, min=3.6491e-01
[AC ] input std=1.8567e+01, recon std=5.9596e+00


1999: AC 277.8782:   7%|▋         | 1993/30598 [00:13<02:31, 189.37it/s]

[AC ] latent var mean=7.8073e+03, min=3.4647e-01
[AC ] input std=1.6948e+01, recon std=6.1636e+00


2999: AC 368.7260:  10%|▉         | 2989/30598 [00:19<02:21, 194.83it/s]

...

29019: AC 183.3245:  95%|█████████▍| 29006/30598 [03:01<00:10, 158.95it/s]

[AC ] latent var mean=1.1618e+05, min=5.7804e-02
[AC ] input std=1.3536e+01, recon std=4.2334e+00


30015: AC 128.6266:  98%|█████████▊| 30012/30598 [03:06<00:03, 152.12it/s]

[AC ] latent var mean=8.7271e+05, min=4.5761e+01
[AC ] input std=2.1250e+01, recon std=6.3229e+00


30595: AC 327.0703: 100%|██████████| 30598/30598 [03:10<00:00, 160.80it/s]
